# Exploration du CSV avant migration

**Projet 5 – Migration des données médicales vers MongoDB**

Avant d'écrire le script de migration, on regarde les données pour savoir ce qu'il faudra contrôler et nettoyer.
L'énoncé demande de tester l'intégrité **avant** la migration : colonnes, types, doublons, valeurs manquantes.

> Lancer les cellules dans l'ordre avec **Maj + Entrée**. Le notebook doit être ouvert depuis la racine du projet (noyau : `venv`).

## 1. Charger le fichier

`pandas` lit le CSV et le range dans un **DataFrame** (un tableau avec des lignes et des colonnes nommées).

In [2]:
import pandas as pd

df = pd.read_csv("../data/healthcare_dataset.csv")

## 2. Taille du tableau

`shape` renvoie `(nombre de lignes, nombre de colonnes)`.

In [3]:
df.shape

(55500, 15)

## 3. Les colonnes

On vérifie que les 15 colonnes attendues sont présentes.

In [4]:
df.columns.tolist()

['Name',
 'Age',
 'Gender',
 'Blood Type',
 'Medical Condition',
 'Date of Admission',
 'Doctor',
 'Hospital',
 'Insurance Provider',
 'Billing Amount',
 'Room Number',
 'Admission Type',
 'Discharge Date',
 'Medication',
 'Test Results']

## 4. Les premières lignes

`head()` affiche les 5 premières lignes. On remarque déjà :
- les **noms** ont des majuscules et minuscules au hasard (`Bobby JacksOn`) ;
- certains **hôpitaux** se terminent par une virgule (`Hernandez Rogers and Vang,`).

In [5]:
df.head()

,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal
1,LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
2,DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal
3,andrEw waTtS,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal
4,adrIENNE bEll,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal


## 5. Les types

`dtypes` montre comment pandas a interprété chaque colonne :
- `int64` = entier, `float64` = décimal, `object` = **texte**.

Les deux colonnes de dates sont en `object` : ce sont du texte, il faudra les convertir en vraies dates
(sinon impossible de chercher par période).

In [6]:
df.dtypes

Name                   object
Age                     int64
Gender                 object
Blood Type             object
Medical Condition      object
Date of Admission      object
Doctor                 object
Hospital               object
Insurance Provider     object
Billing Amount        float64
Room Number             int64
Admission Type         object
Discharge Date         object
Medication             object
Test Results           object
dtype: object

## 6. Valeurs manquantes

`isna()` met `True` là où une case est vide ; `.sum()` compte les `True` par colonne.

In [7]:
df.isna().sum()

Name                  0
Age                   0
Gender                0
Blood Type            0
Medical Condition     0
Date of Admission     0
Doctor                0
Hospital              0
Insurance Provider    0
Billing Amount        0
Room Number           0
Admission Type        0
Discharge Date        0
Medication            0
Test Results          0
dtype: int64

## 7. Doublons

`duplicated()` repère les lignes **strictement identiques** à une ligne déjà vue (le même séjour saisi deux fois).

In [8]:
df.duplicated().sum()

np.int64(534)

Un exemple de doublon : on affiche toutes les occurrences d'une ligne dupliquée.

In [9]:
premier_doublon = df[df.duplicated()].iloc[0]
df[(df == premier_doublon).all(axis=1)]

,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
6319,Samuel joYCe,56,Male,O+,Arthritis,2022-11-03,Krista Hartman,Thomas-Ballard,UnitedHealthcare,15654.830587,157,Urgent,2022-11-22,Aspirin,Inconclusive
50023,Samuel joYCe,56,Male,O+,Arthritis,2022-11-03,Krista Hartman,Thomas-Ballard,UnitedHealthcare,15654.830587,157,Urgent,2022-11-22,Aspirin,Inconclusive


## 8. La casse des noms

Le même patient pourrait être écrit de plusieurs façons : il faudra **normaliser** (`str.title()` → `Bobby Jackson`).

In [10]:
df["Name"].head(10)

0         Bobby JacksOn
1          LesLie TErRy
2           DaNnY sMitH
3          andrEw waTtS
4         adrIENNE bEll
5         EMILY JOHNSOn
6        edwArD EDWaRDs
7    CHrisTInA MARtinez
8       JASmINe aGuIlaR
9      ChRISTopher BerG
Name: Name, dtype: object

In [11]:
df["Name"].head(10).str.title()

0         Bobby Jackson
1          Leslie Terry
2           Danny Smith
3          Andrew Watts
4         Adrienne Bell
5         Emily Johnson
6        Edward Edwards
7    Christina Martinez
8       Jasmine Aguilar
9      Christopher Berg
Name: Name, dtype: object

## 9. Montants négatifs

Une facture négative est inhabituelle (probablement un avoir / remboursement). On ne les supprime pas, on les **signale**.

In [12]:
(df["Billing Amount"] < 0).sum()

np.int64(108)

In [13]:
df.loc[df["Billing Amount"] < 0, ["Name", "Billing Amount", "Insurance Provider"]].head()

,Name,Billing Amount,Insurance Provider
132,ashLEy ERIcKSoN,-502.507813,Aetna
799,CHRisTOPHer wEiss,-1018.245371,Aetna
1018,AsHley WaRnER,-306.364925,Aetna
1421,JAY galloWaY,-109.097122,Blue Cross
2103,josHUa wilLIamSon,-576.727907,Blue Cross


## 10. Bilan : ce que le script de migration devra faire

| Constat | Action dans `src/transform.py` |
|---|---|
| 55 500 lignes, 15 colonnes | vérifier la présence des 15 colonnes (`check_columns`) |
| dates lues comme du texte | convertir en `datetime` |
| aucune valeur manquante | le contrôler quand même (`integrity_report`) |
| 534 doublons exacts | les supprimer (`drop_duplicates`) |
| noms en casse aléatoire | normaliser avec `str.title()` |
| virgule finale sur certains hôpitaux | la retirer |
| 108 montants négatifs | les signaler, arrondir les montants à 2 décimales |